In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

In [ ]:
data_path = Path("../data")

creditcardfraud = data_path / "creditcardfraud/creditcard.csv"


In [ ]:
df = pd.read_csv(creditcardfraud)

In [ ]:
X = df.drop('Class', axis=1)
y = df['Class']

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print(f"Размер train: {X_train.shape}, val: {X_val.shape}")
print(f"Мошенников в train: {y_train.mean():.4f}")
print(f"Мошенников в val: {y_val.mean():.4f}")

In [ ]:
X_train

In [ ]:
# # Можем отмасштабировать Amount
# scaler = StandardScaler()
# X_train['Amount'] = scaler.fit_transform(X_train[['Amount']])
# X_val['Amount'] = scaler.transform(X_val[['Amount']])

In [ ]:
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

results = {}

# --- XGBoost ---
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=len(y_train[y_train==0]) / len(y_train[y_train==1]),
    random_state=42,
    eval_metric='logloss'
)
xgb_model.fit(X_train, y_train)
y_pred_proba_xgb = xgb_model.predict_proba(X_val)[:, 1]
results['XGBoost'] = y_pred_proba_xgb

# --- LightGBM ---
lgb_model = LGBMClassifier(
    n_estimators=20,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=len(y_train[y_train==0]) / len(y_train[y_train==1]),
    random_state=42,
    verbose=-1
)
lgb_model.fit(X_train, y_train)
y_pred_proba_lgb = lgb_model.predict_proba(X_val)[:, 1]
results['LightGBM'] = y_pred_proba_lgb

# --- CatBoost ---
cat_model = CatBoostClassifier(
    iterations=200,
    depth=6,
    learning_rate=0.1,
    auto_class_weights='Balanced',
    random_seed=42,
    verbose=False
)
cat_model.fit(X_train, y_train)
y_pred_proba_cat = cat_model.predict_proba(X_val)[:, 1]
results['CatBoost'] = y_pred_proba_cat

In [ ]:
for model_name, y_pred_proba in results.items():
    y_pred_binary = (y_pred_proba > 0.5).astype(int)
    
    roc_auc = roc_auc_score(y_val, y_pred_proba)
    pr_auc = average_precision_score(y_val, y_pred_proba)
    f1 = f1_score(y_val, y_pred_binary)
    
    print(f"\n{model_name}:")
    print(f"  ROC-AUC:  {roc_auc:.4f}")
    print(f"  PR-AUC:   {pr_auc:.4f}")
    print(f"  F1-score: {f1:.4f}")

In [ ]:
# Оптимизация гиперпараметров

In [ ]:

import optuna
from optuna.samplers import TPESampler


from imblearn.over_sampling import SMOTE


In [ ]:
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print(f"Размер train после SMOTE: {X_train_resampled.shape}")
print(f"Доля мошенничества в train после SMOTE: {y_train_resampled.mean():.4f}")

In [ ]:
N_TRIALS = 30  # Количество итераций для оптимизации

# --- XGBoost ---
def objective_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1, 10),
        'random_state': 42,
        'use_label_encoder': False,
        'eval_metric': 'logloss'
    }
    model = XGBClassifier(**params)
    model.fit(X_train_resampled, y_train_resampled)
    preds = model.predict_proba(X_val)[:, 1]
    return average_precision_score(y_val, preds)

print("\nОптимизация XGBoost...")
study_xgb = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
study_xgb.optimize(objective_xgb, n_trials=N_TRIALS, show_progress_bar=True)
best_xgb_params = study_xgb.best_params
print(f"Лучшие параметры XGBoost: {best_xgb_params}")
print(f"Лучший PR-AUC: {study_xgb.best_value:.4f}")

# --- LightGBM ---
def objective_lgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'num_leaves': trial.suggest_int('num_leaves', 10, 50),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1, 10),
        'random_state': 42,
        'verbose': -1
    }
    model = LGBMClassifier(**params)
    model.fit(X_train_resampled, y_train_resampled)
    preds = model.predict_proba(X_val)[:, 1]
    return average_precision_score(y_val, preds)

print("\nОптимизация LightGBM...")
study_lgb = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
study_lgb.optimize(objective_lgb, n_trials=N_TRIALS, show_progress_bar=True)
best_lgb_params = study_lgb.best_params
print(f"Лучшие параметры LightGBM: {best_lgb_params}")
print(f"Лучший PR-AUC: {study_lgb.best_value:.4f}")

# --- CatBoost ---
def objective_cat(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 100, 500),
        'depth': trial.suggest_int('depth', 3, 9),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True),
        'random_seed': 42,
        'auto_class_weights': 'Balanced',
        'verbose': False
    }
    model = CatBoostClassifier(**params)
    model.fit(X_train_resampled, y_train_resampled)
    preds = model.predict_proba(X_val)[:, 1]
    return average_precision_score(y_val, preds)

print("\nОптимизация CatBoost...")
study_cat = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
study_cat.optimize(objective_cat, n_trials=N_TRIALS, show_progress_bar=True)
best_cat_params = study_cat.best_params
print(f"Лучшие параметры CatBoost: {best_cat_params}")
print(f"Лучший PR-AUC: {study_cat.best_value:.4f}")

In [ ]:
xgb_model = XGBClassifier(**best_xgb_params, random_state=42, use_label_encoder=False, eval_metric='logloss')
xgb_model.fit(X_train_resampled, y_train_resampled)

# LightGBM
lgb_model = LGBMClassifier(**best_lgb_params, random_state=42, verbose=-1)
lgb_model.fit(X_train_resampled, y_train_resampled)

# CatBoost
cat_model = CatBoostClassifier(**best_cat_params, random_seed=42, auto_class_weights='Balanced', verbose=False)
cat_model.fit(X_train_resampled, y_train_resampled)

models = {
    'XGBoost': xgb_model,
    'LightGBM': lgb_model,
    'CatBoost': cat_model
}

In [ ]:
print(5 * "=" + "xgboost params" + 5 * "=")
for name, val in best_xgb_params.items():
    print(name, round(val, 3))

print(5 * "=" + "lightgbm params" + 5 * "=")
for name, val in best_lgb_params.items():
    print(name, round(val, 3))

print(5 * "=" + "catboost params" + 5 * "=")
for name, val in best_cat_params.items():
    print(name, round(val, 3))

In [ ]:
for name, model in models.items():
    pred_proba = model.predict_proba(X_val)[:, 1]
    pred_binary = (pred_proba > 0.5).astype(int)
    print(f"\n{name}:")
    print(f"  ROC-AUC:  {roc_auc_score(y_val, pred_proba):.4f}")
    print(f"  PR-AUC:   {average_precision_score(y_val, pred_proba):.4f}")
    print(f"  F1:       {f1_score(y_val, pred_binary):.4f}")

In [ ]:
# Анализ важности

In [ ]:

import shap
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
for name, model in models.items():
    print(f"\n--- SHAP анализ для {name} ---")
    
    # Используем TreeExplainer для деревьев (быстро и точно)[reference:7]
    explainer = shap.TreeExplainer(model)
    # Для SHAP нужна выборка, берем часть валидационной (первые 1000 для скорости)
    X_sample = X_val.sample(n=min(1000, len(X_val)), random_state=42)
    shap_values = explainer.shap_values(X_sample)
    
    # 1. Summary Plot (главный график)
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values, X_sample, show=False)
    plt.title(f'SHAP Summary Plot - {name}')
    plt.tight_layout()
    plt.show()
    
    # 2. Bar Plot (усредненная важность признаков)[reference:8]
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values, X_sample, plot_type="bar", show=False)
    plt.title(f'SHAP Feature Importance (Bar) - {name}')
    plt.tight_layout()
    plt.show()

In [ ]:
for name, model in models.items():
    if hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
        feature_names = X_train.columns
        imp_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
        imp_df = imp_df.sort_values('importance', ascending=False).head(15)
        
        plt.figure(figsize=(10, 6))
        sns.barplot(data=imp_df, x='importance', y='feature')
        plt.title(f'Feature Importance (built-in) - {name}')
        plt.tight_layout()
        plt.show()

In [ ]:
# Оценка bias / variance после обучения

In [ ]:


# Пример для XGBoost (перед fit)
eval_set = [(X_train_resampled, y_train_resampled), (X_val, y_val)]
xgb_model.fit(X_train_resampled, y_train_resampled, eval_set=eval_set, verbose=False)
results = xgb_model.evals_result()

# Отрисовка
epochs = len(results['validation_0']['logloss'])
train_loss = results['validation_0']['logloss']
val_loss = results['validation_1']['logloss']

plt.plot(epochs, train_loss, label='Train Loss')
plt.plot(epochs, val_loss, label='Val Loss')
plt.legend(); plt.show()